In [ ]:
%pip install -q torch-geometric pyarrow

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/BD_LAB2")
DATA_DIR = Path("/content/drive/MyDrive/BD_LAB2/db")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATA_DIR)


In [ ]:
import pandas as pd

issues_comments = pd.read_parquet(DATA_DIR / "IssueComments.parquet")
issues = pd.read_parquet(DATA_DIR / "Issues.parquet")
pr_review_comments = pd.read_parquet(DATA_DIR / "PRReviewComments.parquet")
pr_reviews = pd.read_parquet(DATA_DIR / "PRReviews.parquet")
pull_requests = pd.read_parquet(DATA_DIR / "PullRequests.parquet")
pushes = pd.read_parquet(DATA_DIR / "Pushes.parquet")
reposetories = pd.read_parquet(DATA_DIR / "Repositories.parquet")
users = pd.read_parquet(DATA_DIR / "Users.parquet")

num_users = len(users)
num_repos = len(reposetories)
num_issues = len(issues)
num_prs = len(pull_requests)
num_pushes = len(pushes)
num_pr_reviews = len(pr_reviews)
num_pr_review_comments = len(pr_review_comments)
num_issue_comments = len(issues_comments)

print("issue_comments shape:", issues_comments.shape)
print("issues shape:", issues.shape)
print("pr_review_comments shape:", pr_review_comments.shape)
print("pr_reviews shape:", pr_reviews.shape)
print("pull_requests shape:", pull_requests.shape)
print("pushes shape:", pushes.shape)
print("reposetories shape:", reposetories.shape)


In [ ]:
from torch_geometric.nn import GCNConv

class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x


In [ ]:
cutoff_date = pd.Timestamp("2026-10-25")
week_end = cutoff_date + pd.Timedelta(days=7)

past_pushes = pushes[pushes["created_at"] < cutoff_date]

week_pushes = pushes[  # Filter pushes that occurred within the week after the cutoff date
    (pushes["created_at"] >= cutoff_date) &
    (pushes["created_at"] < week_end)
]

past_pull_requests = pull_requests[pull_requests["first_seen_at"] < cutoff_date]

week_pull_requests = pull_requests[  # Filter pull requests that occurred within the week after the cutoff date
    (pull_requests["created_at"] >= cutoff_date) &
    (pull_requests["created_at"] < week_end)
]

past_pr_reviews = pr_reviews[pr_reviews["first_seen_at"] < cutoff_date]

week_pr_reviews = pr_reviews[  # Filter pull request reviews that occurred within the week after the cutoff date
    (pr_reviews["submitted_at"] >= cutoff_date) &
    (pr_reviews["submitted_at"] < week_end)
]

past_issues = issues[issues["first_seen_at"] < cutoff_date]

week_issues = issues[
    (issues["created_at"] >= cutoff_date) &
    (issues["created_at"] < week_end) &
    (~issues["is_pull_request"])
]

past_issue_comments = issues_comments[issues_comments["created_at"] < cutoff_date]

week_issue_comments = issues_comments[  # Filter issue comments that occurred within the week after the cutoff date
    (issues_comments["created_at"] >= cutoff_date) &
    (issues_comments["created_at"] < week_end)
]

past_pr_review_comments = pr_review_comments[pr_review_comments["created_at"] < cutoff_date]

week_pr_review_comments = pr_review_comments[  # Filter pull request review comments that occurred within the week after the cutoff date
    (pr_review_comments["created_at"] >= cutoff_date) &
    (pr_review_comments["created_at"] < week_end)
]

# Train on cohort users already observed at the prediction cutoff.
targets = users.loc[
    users["is_cohort"] & (users["first_seen_at"] < cutoff_date),
    ["user_id"],
].copy()


In [ ]:

pushed_this_week = targets["user_id"].isin(week_pushes["user_id"]) # Check if each target user pushed within the week after the cutoff date
targets["push"] = pushed_this_week.astype(int) # Add a new column "push" to the targets DataFrame, where 1 indicates the user pushed and 0 indicates they did not push

pull_requests_this_week = targets["user_id"].isin(week_pull_requests["author_id"]) # Check if each target user created a pull request within the week after the cutoff date
targets["open_pr"] = pull_requests_this_week.astype(int) # Add a new column "

issue_this_week = targets["user_id"].isin(week_issues["author_id"]) # Check if each target user created an issue within the week after the cutoff date
targets["open_issue"] = issue_this_week.astype(int) # Add a new column "issue"

issue_comment_this_week = targets["user_id"].isin(week_issue_comments["author_id"]) # Check if each target user created an issue comment within the week after the cutoff date
targets["issue_comment"] = issue_comment_this_week.astype(int) # Add a new column "

pr_review_this_week = targets["user_id"].isin(week_pr_reviews["reviewer_id"]) # Check if each target user created a pull request review within the week after the cutoff date
targets["pr_review"] = pr_review_this_week.astype(int) # Add a new column "

pr_review_comment_this_week = targets["user_id"].isin(week_pr_review_comments["author_id"]) # Check if each target user created a pull request review comment within the week after the cutoff date
targets["pr_review_comment"] = pr_review_comment_this_week.astype(int) # Add a new


In [ ]:
user_nodes = torch.tensor(
    past_pushes["user_id"].to_numpy(), dtype=torch.long
)
repo_nodes = torch.tensor(
    past_pushes["repo_id"].to_numpy() + num_users, dtype=torch.long
)

push_edges = torch.stack([user_nodes, repo_nodes], dim=0)

print(push_edges.shape)
print(push_edges[:, :5])

reverse_push_edges = push_edges.flip(0)

edge_index = torch.cat([push_edges, reverse_push_edges], dim=1)

print(edge_index.shape)
print(edge_index[:, :5])

In [ ]:
pr_links = past_pull_requests[["pr_id", "author_id", "repo_id"]].dropna()

pr_nodes = torch.tensor(
    pr_links["pr_id"].to_numpy(dtype=np.int64) + num_users + num_repos,
    dtype=torch.long,
)

pr_repo_nodes = torch.tensor(
    pr_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
    dtype=torch.long,
)

pr_author_nodes = torch.tensor(
    pr_links["author_id"].to_numpy(dtype=np.int64),
    dtype=torch.long,
)

pr_to_author = torch.stack([pr_nodes, pr_author_nodes], dim=0)
pr_to_repo = torch.stack([pr_nodes, pr_repo_nodes], dim=0)

pr_edges = torch.cat([pr_to_author, pr_to_repo], dim=1)
reverse_pr_edges = pr_edges.flip(0)
pr_edges = torch.cat([pr_edges, reverse_pr_edges], dim=1)

edge_index = torch.cat([edge_index, pr_edges], dim=1)

In [ ]:
issue_offset = num_users + num_repos + num_prs

issue_author_links = past_issues[["issue_id", "author_id"]].dropna()
issue_to_author = torch.stack([
    torch.tensor(
        issue_author_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_author_links["author_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

issue_repo_links = past_issues[["issue_id", "repo_id"]].dropna()
issue_to_repo = torch.stack([
    torch.tensor(
        issue_repo_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

issue_assignee_links = past_issues[["issue_id", "assignee_id"]].dropna()
issue_to_assignee = torch.stack([
    torch.tensor(
        issue_assignee_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_assignee_links["assignee_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

issue_pr_links = past_issues[["issue_id", "pull_request_id"]].dropna()
issue_to_pr = torch.stack([
    torch.tensor(
        issue_pr_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_pr_links["pull_request_id"].to_numpy(dtype=np.int64)
        + num_users + num_repos,
        dtype=torch.long,
    ),
], dim=0)

issue_edges = torch.cat([
    issue_to_author, issue_to_repo, issue_to_assignee, issue_to_pr,
], dim=1)
reverse_issue_edges = issue_edges.flip(0)
issue_edges = torch.cat([issue_edges, reverse_issue_edges], dim=1)

edge_index = torch.cat([edge_index, issue_edges], dim=1)

print("Issue edges including reverse connections:", issue_edges.shape)


In [ ]:
# PR review nodes follow the issue nodes.
review_offset = issue_offset + num_issues

review_to_reviewer_links = past_pr_reviews[["review_id", "reviewer_id"]].dropna()
review_to_reviewer = torch.stack([
    torch.tensor(
        review_to_reviewer_links["review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        review_to_reviewer_links["reviewer_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

review_to_repo_links = past_pr_reviews[["review_id", "repo_id"]].dropna()
review_to_repo = torch.stack([
    torch.tensor(
        review_to_repo_links["review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        review_to_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

review_to_pr_links = past_pr_reviews[["review_id", "pr_id"]].dropna()
review_to_pr = torch.stack([
    torch.tensor(
        review_to_pr_links["review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        review_to_pr_links["pr_id"].to_numpy(dtype=np.int64) + num_users + num_repos,
        dtype=torch.long,
    ),
], dim=0)

review_edges = torch.cat([
    review_to_reviewer, review_to_repo, review_to_pr,
], dim=1)
reverse_review_edges = review_edges.flip(0)
review_edges = torch.cat([review_edges, reverse_review_edges], dim=1)

edge_index = torch.cat([edge_index, review_edges], dim=1)

print("Edges including reverse connections:", review_edges.shape)


In [ ]:
# Issue comment nodes follow the PR review nodes.
issue_comment_offset = review_offset + num_pr_reviews

issue_comment_to_author_links = past_issue_comments[["comment_id", "author_id"]].dropna()
issue_comment_to_author = torch.stack([
    torch.tensor(
        issue_comment_to_author_links["comment_id"].to_numpy(dtype=np.int64) + issue_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_comment_to_author_links["author_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

issue_comment_to_repo_links = past_issue_comments[["comment_id", "repo_id"]].dropna()
issue_comment_to_repo = torch.stack([
    torch.tensor(
        issue_comment_to_repo_links["comment_id"].to_numpy(dtype=np.int64) + issue_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_comment_to_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

issue_comment_to_issue_links = past_issue_comments[["comment_id", "issue_id"]].dropna()
issue_comment_to_issue = torch.stack([
    torch.tensor(
        issue_comment_to_issue_links["comment_id"].to_numpy(dtype=np.int64) + issue_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_comment_to_issue_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
], dim=0)

issue_comment_edges = torch.cat([
    issue_comment_to_author, issue_comment_to_repo, issue_comment_to_issue,
], dim=1)
reverse_issue_comment_edges = issue_comment_edges.flip(0)
issue_comment_edges = torch.cat([issue_comment_edges, reverse_issue_comment_edges], dim=1)

edge_index = torch.cat([edge_index, issue_comment_edges], dim=1)

print("Edges including reverse connections:", issue_comment_edges.shape)


In [ ]:
# PR review comment nodes follow the issue comment nodes.
pr_review_comment_offset = issue_comment_offset + num_issue_comments

pr_review_comment_to_author_links = past_pr_review_comments[["comment_id", "author_id"]].dropna()
pr_review_comment_to_author = torch.stack([
    torch.tensor(
        pr_review_comment_to_author_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_author_links["author_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_repo_links = past_pr_review_comments[["comment_id", "repo_id"]].dropna()
pr_review_comment_to_repo = torch.stack([
    torch.tensor(
        pr_review_comment_to_repo_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_pr_links = past_pr_review_comments[["comment_id", "pr_id"]].dropna()
pr_review_comment_to_pr = torch.stack([
    torch.tensor(
        pr_review_comment_to_pr_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_pr_links["pr_id"].to_numpy(dtype=np.int64) + num_users + num_repos,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_review_links = past_pr_review_comments[["comment_id", "pull_request_review_id"]].dropna()
pr_review_comment_to_review = torch.stack([
    torch.tensor(
        pr_review_comment_to_review_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_review_links["pull_request_review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_parent_links = past_pr_review_comments[["comment_id", "in_reply_to_id"]].dropna()
pr_review_comment_to_parent = torch.stack([
    torch.tensor(
        pr_review_comment_to_parent_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_parent_links["in_reply_to_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_edges = torch.cat([
    pr_review_comment_to_author, pr_review_comment_to_repo, pr_review_comment_to_pr, pr_review_comment_to_review, pr_review_comment_to_parent,
], dim=1)
reverse_pr_review_comment_edges = pr_review_comment_edges.flip(0)
pr_review_comment_edges = torch.cat([pr_review_comment_edges, reverse_pr_review_comment_edges], dim=1)

edge_index = torch.cat([edge_index, pr_review_comment_edges], dim=1)

print("Edges including reverse connections:", pr_review_comment_edges.shape)


In [ ]:
total_nodes = (
    num_users + num_repos + num_prs + num_issues
    + num_pr_reviews + num_issue_comments + num_pr_review_comments
)

x = torch.zeros((total_nodes, 7), dtype=torch.float32)

boundaries = [
    0,
    num_users,
    num_users + num_repos,
    num_users + num_repos + num_prs,
    num_users + num_repos + num_prs + num_issues,
    num_users + num_repos + num_prs + num_issues + num_pr_reviews,
    num_users + num_repos + num_prs + num_issues + num_pr_reviews + num_issue_comments,
    total_nodes,
]

for node_type in range(7):
    x[boundaries[node_type]:boundaries[node_type + 1], node_type] = 1

print("Node features:", x.shape)
print("Graph edges:", edge_index.shape)

In [ ]:
from torch_geometric.data import Data

graph = Data(x=x, edge_index=edge_index)

print(graph)
print("Highest node number in edges:", edge_index.max().item())
print("Number of feature rows:", x.shape[0])

assert edge_index.max().item() < x.shape[0]

In [ ]:
activity_columns = [
    "open_pr", "open_issue", "issue_comment",
    "pr_review", "pr_review_comment", "push",
]

assert np.array_equal(users["user_id"].to_numpy(), np.arange(num_users))

target_nodes = torch.tensor(
    targets["user_id"].to_numpy(dtype=np.int64),
    dtype=torch.long,
)
y = torch.tensor(
    targets[activity_columns].to_numpy(dtype=np.float32),
    dtype=torch.float32,
)

print("User nodes to predict:", target_nodes.shape)
print("Their six answers:", y.shape)

In [ ]:
# 1. Create the model and put its inputs on the same device.
# Each node starts with 7 features, gets 64 hidden features,
# and finishes with 6 activity scores.
graph = graph.to(device)
target_nodes = target_nodes.to(device)
y = y.to(device)

model = GCN(
    input_dim=graph.x.shape[1],
    hidden_dim=64,
    output_dim=y.shape[1],
).to(device)

print(model)
print("Training device:", device)


In [ ]:
# 2. Set up the loss and optimizer.
# The loss compares each raw activity score with its 0/1 target.
criterion = nn.BCEWithLogitsLoss()

# Adam updates the model weights. lr controls the update step size.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# 3. Train on the historical graph.
# Rerun the model and optimizer cells to start a fresh training run.
num_epochs = 100
training_losses = []

for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()  # Clear gradients from the previous step.

    all_node_scores = model(graph)
    user_scores = all_node_scores[target_nodes]  # Match the row order in y.
    assert user_scores.shape == y.shape

    loss = criterion(user_scores, y)
    if not torch.isfinite(loss):
        raise RuntimeError("Training loss is not finite.")

    loss.backward()  # Calculate how each weight affects the loss.
    optimizer.step()  # Update the weights using those gradients.

    training_losses.append(loss.item())
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d}/{num_epochs} | Training loss: {loss.item():.4f}")

# Later predictions should use evaluation mode to disable dropout.
model.eval()
print("Training finished. Validation on a later week is the next step.")


In [ ]:
# Save the trained weights and the settings needed to reload them.
OUTPUT_DIR = PROJECT_DIR / "training_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = OUTPUT_DIR / "gcn_checkpoint.pt"

checkpoint = {
    "model_state_dict": {
        name: weight.detach().cpu()
        for name, weight in model.state_dict().items()
    },
    "model_config": {
        "input_dim": graph.x.shape[1],
        "hidden_dim": model.conv1.out_channels,
        "output_dim": y.shape[1],
    },
    "activity_columns": activity_columns,
    "node_type_order": [
        "users",
        "repositories",
        "pull_requests",
        "issues",
        "pr_reviews",
        "issue_comments",
        "pr_review_comments",
    ],
    "training_cutoff": str(cutoff_date),
    "training_week_end": str(week_end),
    "epochs_completed": len(training_losses),
    "training_losses": training_losses,
}

torch.save(checkpoint, checkpoint_path)
print("Model saved to:", checkpoint_path)